# Benchmark: 6D Reconstruction (GPSR Notebook)
This notebook benchmarks the execution speed of the 6D reconstruction workflow under two Cloud-in-Cell (CIC) implementations:
1. **Current Optimized Version**: The tensorised implementation in the active codebase.
2. **Chenran's Loop-based Version**: The loop-based implementation right after Chenran's fixes (commit `c3cd873a`).

You can select the hardware accelerator (CPU, MPS, CUDA) at the top of the notebook. The results are saved to `dev/benchmark_results` to compare performance across different nodes (e.g. Apple M1 Pro and A100 GPU nodes).

In [ ]:
# Device selection: change this to 'cpu', 'cuda', or 'mps'
device = "cpu"
max_epochs = 20  # You can increase this to 100 or 500 for a longer benchmark

In [ ]:
import sys
import os
import time
import re
import glob
import platform
import subprocess
import pandas as pd
import numpy as np
import torch
import lightning as L

# Set repository root relative to dev/
repo_root = os.path.abspath("..")
if repo_root not in sys.path:
    sys.path.append(repo_root)

print("Imports completed successfully.")

In [ ]:
def get_cpu_info():
    try:
        if platform.system() == "Windows":
            return platform.processor()
        elif platform.system() == "Darwin":
            return subprocess.check_output(["sysctl", "-n", "machdep.cpu.brand_string"]).decode().strip()
        elif platform.system() == "Linux":
            with open("/proc/cpuinfo", "r") as f:
                for line in f:
                    if "model name" in line:
                        return re.sub(".*model name.*:", "", line, 1).strip()
    except Exception:
        pass
    return platform.processor() or "Unknown CPU"

def get_ram_info():
    try:
        if platform.system() == "Darwin":
            mem_bytes = int(subprocess.check_output(["sysctl", "-n", "hw.memsize"]).decode().strip())
            return f"{mem_bytes / (1024**3):.1f} GB"
        elif platform.system() == "Linux":
            with open("/proc/meminfo", "r") as f:
                for line in f:
                    if "MemTotal" in line:
                        mem_kb = int(line.split()[1])
                        return f"{mem_kb / (1024**2):.1f} GB"
    except Exception:
        pass
    return "Unknown RAM"

def get_gpu_info():
    gpus = []
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            gpus.append(torch.cuda.get_device_name(i))
    if torch.backends.mps.is_available():
        gpus.append("Apple Silicon Integrated GPU (MPS)")
    return ", ".join(gpus) if gpus else "None"

cpu_info = get_cpu_info()
ram_info = get_ram_info()
gpu_info = get_gpu_info()
os_info = f"{platform.system()} {platform.release()}"

system_label = f"{os_info} | {cpu_info} | RAM: {ram_info}"
if gpu_info != "None":
    system_label += f" | GPU: {gpu_info}"

print("System Information:")
print(f"OS:  {os_info}")
print(f"CPU: {cpu_info}")
print(f"RAM: {ram_info}")
print(f"GPU: {gpu_info}")
print(f"Label: {system_label}")

In [ ]:
from gpsr.datasets import split_dataset
from gpsr.modeling import GPSR6DLattice, GPSR
from gpsr.train import LitGPSR
from gpsr.beams import NNParticleBeamGenerator

# Path to the synthetic 6D dataset. Adjust this if the gpsr repo is cloned elsewhere on your A100 node.
dset_path = os.path.expanduser("~/Documents/DESY/gpsr/docs/examples/example_data/example_datasets/reconstruction_6D.dset")
print(f"Loading dataset from {dset_path}...")
dset = torch.load(dset_path, weights_only=False)

train_k_ids = np.arange(0, len(dset.six_d_parameters), 2)
train_dset, test_dset = split_dataset(dset, train_k_ids)
train_loader = torch.utils.data.DataLoader(train_dset, batch_size=10)

p0c = 43.36e6  # reference momentum in eV/c
screens = train_dset.screens
l_quad = 0.11
l_tdc = 0.01
f_tdc = 1.3e9
phi_tdc = 0.0
l_bend = 0.3018
theta_on = -20.0 * 3.14 / 180.0
l1 = 0.790702
l2 = 0.631698
l3 = 0.889

gpsr_lattice = GPSR6DLattice(
    l_quad, l_tdc, f_tdc, phi_tdc, l_bend, theta_on, l1, l2, l3, *screens
)
print("Dataset loaded and GPSR6DLattice initialized.")

In [ ]:
# Monkeypatch set_lattice_parameters to clear cheetah's cache, avoiding shape mismatch errors
def set_lattice_parameters_with_cache_clearing(self, x: torch.Tensor) -> None:
    for elem in self.segment.elements:
        if hasattr(elem, "_cache"):
            elem._cache.clear()
        if hasattr(elem, "elements"):
            for sub_elem in elem.elements:
                if hasattr(sub_elem, "_cache"):
                    sub_elem._cache.clear()
    self.segment.SCAN_QUAD.k1.data = x[..., 0]
    self.segment.SCAN_TDC.voltage.data = x[..., 1]
    G = x[..., 2]
    bend_angle = torch.arcsin(self.l_bend * G)
    arc_length = bend_angle / G
    self.segment.SCAN_DIPOLE.angle.data = bend_angle
    self.segment.SCAN_DIPOLE.length.data = arc_length
    self.segment.SCAN_DIPOLE.dipole_e2.data = bend_angle
    self.segment.DIPOLE_TO_SCREEN.length.data = (
        self.l3 - self.l_bend / 2 / torch.cos(bend_angle)
    )

GPSR6DLattice.set_lattice_parameters = set_lattice_parameters_with_cache_clearing
print("Cache-clearing parameter update hook registered!")

In [ ]:
# Warmup pass to initialize GPU/device contexts and trigger PyTorch JIT warmups
print("Running warmup pass to initialize GPU/device contexts...")
torch.manual_seed(42)
np.random.seed(42)
L.seed_everything(42)

# Create a dummy model and trainer for warmup
gpsr_model_warmup = GPSR(NNParticleBeamGenerator(10000, p0c), gpsr_lattice)
litgpsr_warmup = LitGPSR(gpsr_model_warmup)

accelerator = "gpu" if device in ["cuda", "mps"] else "cpu"
devices = [0] if device in ["cuda", "mps"] else 1

trainer_warmup = L.Trainer(
    max_epochs=2,
    accelerator=accelerator,
    devices=devices,
    enable_checkpointing=False,
    logger=False
)
trainer_warmup.fit(model=litgpsr_warmup, train_dataloaders=train_loader)
print("Warmup pass completed successfully.\n")

print(f"--- RUN 1: Current Optimized Version on {device.upper()} ---")
# Reset seed for exact initialization
torch.manual_seed(42)
np.random.seed(42)
L.seed_everything(42)

# Clear any remaining cache
for elem in gpsr_lattice.segment.elements:
    if hasattr(elem, "_cache"):
        elem._cache.clear()
    if hasattr(elem, "elements"):
        for sub_elem in elem.elements:
            if hasattr(sub_elem, "_cache"):
                sub_elem._cache.clear()

gpsr_model_opt = GPSR(NNParticleBeamGenerator(10000, p0c), gpsr_lattice)
litgpsr_opt = LitGPSR(gpsr_model_opt)

trainer_opt = L.Trainer(
    max_epochs=max_epochs,
    accelerator=accelerator,
    devices=devices,
    enable_checkpointing=False,
    logger=False
)

t0 = time.perf_counter()
trainer_opt.fit(model=litgpsr_opt, train_dataloaders=train_loader)
t_opt = time.perf_counter() - t0
print(f"\nCurrent Optimized execution time: {t_opt:.4f} seconds")

In [ ]:
from typing import Sequence
import cheetah.utils.cloud_in_cell
import cheetah.accelerator.screen

def deposit_charge_cic_chenran(
    positions: Sequence[torch.Tensor],
    bins: Sequence[torch.Tensor],
    weights: torch.Tensor | None = None,
) -> torch.Tensor:
    if not positions:
        raise ValueError("positions list cannot be empty")
    if len(positions) != len(bins):
        raise ValueError("Number of position tensors must match number of bin arrays")
    
    ndim = len(positions)
    first_pos = positions[0]
    device_ = first_pos.device
    dtype = first_pos.dtype

    for i, pos in enumerate(positions[1:], 1):
        if pos.shape != first_pos.shape:
            raise ValueError(
                f"All position tensors must have the same shape. "
                f"positions[0] has shape {first_pos.shape}, "
                f"positions[{i}] has shape {pos.shape}"
            )
        if pos.device != device_:
            raise ValueError("All tensors must be on the same device")
        if pos.dtype != dtype:
            raise ValueError("All tensors must have the same dtype")

    grid_sizes = []
    spacings = []

    for i, bin_array in enumerate(bins):
        N = bin_array.numel()
        if N < 2:
            raise ValueError(f"bins[{i}] must have at least 2 elements")
        spacing = bin_array[1] - bin_array[0]
        if N > 2:
            diffs = torch.diff(bin_array)
            if not torch.allclose(diffs, spacing, rtol=1e-4):
                raise ValueError(f"bins[{i}] must have uniform spacing")
        grid_sizes.append(N)
        spacings.append(spacing)

    if weights is None:
        weights = torch.ones_like(first_pos)

    for pos, bin_array in zip(positions, bins):
        outside_mask = (pos < bin_array[0]) | (pos >= bin_array[-1])
        weights = weights * (~outside_mask).float()

    grid_indices = []
    fractional_parts = []

    for pos, bin_array, spacing in zip(positions, bins, spacings):
        u = (pos - bin_array[0]) / spacing
        i = torch.floor(u).to(torch.int64)
        grid_indices.append(i)
        w = u - i
        fractional_parts.append(w)

    num_corners = 2**ndim
    corner_indices = []
    corner_weights = []

    for corner in range(num_corners):
        corner_offsets = []
        weight_factors = []
        for dim in range(ndim):
            if (corner >> dim) & 1:
                corner_offsets.append(1)
                weight_factors.append(fractional_parts[dim])
            else:
                corner_offsets.append(0)
                weight_factors.append(1 - fractional_parts[dim])

        corner_idx_list = []
        for dim in range(ndim):
            base_idx = grid_indices[dim]
            offset_idx = (base_idx + corner_offsets[dim]).clamp(0, grid_sizes[dim] - 1)
            corner_idx_list.append(offset_idx)

        corner_weight = weights
        for weight_factor in weight_factors:
            corner_weight = corner_weight * weight_factor
        corner_indices.append(corner_idx_list)
        corner_weights.append(corner_weight)

    def multi_to_flat_index(idx_list):
        flat_idx = idx_list[0]
        stride = 1
        for dim in range(1, ndim):
            stride *= grid_sizes[dim - 1]
            flat_idx = flat_idx + idx_list[dim] * stride
        return flat_idx

    batch_shape = first_pos.shape[:-1]
    B = int(torch.tensor(batch_shape).prod()) if batch_shape else 1
    N = first_pos.shape[-1]

    def flatten_tensor(t):
        return t.reshape(B, N)

    all_flat_indices = []
    all_weights = []
    for corner_idx_list, corner_weight in zip(corner_indices, corner_weights):
        flat_idx = multi_to_flat_index(corner_idx_list)
        all_flat_indices.append(flatten_tensor(flat_idx))
        all_weights.append(flatten_tensor(corner_weight))

    idx_all = torch.cat(all_flat_indices, dim=1)
    vals_all = torch.cat(all_weights, dim=1)

    total_grid_size = int(torch.tensor(grid_sizes).prod())
    charge = torch.zeros((B, total_grid_size), dtype=dtype, device=device_)
    for b in range(B):
        charge[b].index_add_(0, idx_all[b], vals_all[b])

    cell_volume = 1.0
    for spacing in spacings:
        cell_volume *= spacing
    inv_cell_volume = 1.0 / cell_volume
    charge = charge * inv_cell_volume

    out_shape = (*batch_shape, *grid_sizes[::-1])
    charge = charge.reshape(out_shape)
    batch_ndim = len(batch_shape)
    spatial_axes = list(range(batch_ndim, batch_ndim + ndim))
    return charge.permute(*range(batch_ndim), *reversed(spatial_axes))

def wrapper_chenran(positions, bins, extent, charges=None):
    positions_list = [positions[..., d] for d in range(positions.shape[-1])]
    
    bins_list = []
    for d in range(positions.shape[-1]):
        lim0, lim1 = extent[d]
        Nd = bins[d] + 1
        bin_array = torch.linspace(lim0, lim1, Nd, device=positions.device, dtype=positions.dtype)
        bins_list.append(bin_array)
        
    out_chenran = deposit_charge_cic_chenran(positions_list, bins_list, charges)
    
    cell_volume = 1.0
    for d in range(positions.shape[-1]):
        lim0, lim1 = extent[d]
        Nd = bins[d]
        cell_volume *= (lim1 - lim0) / Nd
    
    return out_chenran * cell_volume

cheetah.utils.cloud_in_cell.cloud_in_cell_charge_deposition = wrapper_chenran
cheetah.accelerator.screen.cloud_in_cell_charge_deposition = wrapper_chenran
print("Monkeypatched to Chenran loop-based implementation!")

In [ ]:
print("Running warmup pass for Chenran loop-based implementation...")
torch.manual_seed(42)
np.random.seed(42)
L.seed_everything(42)

# Create a dummy model and trainer for warmup
gpsr_model_chenran_warmup = GPSR(NNParticleBeamGenerator(10000, p0c), gpsr_lattice)
litgpsr_chenran_warmup = LitGPSR(gpsr_model_chenran_warmup)

trainer_chenran_warmup = L.Trainer(
    max_epochs=2,
    accelerator=accelerator,
    devices=devices,
    enable_checkpointing=False,
    logger=False
)
trainer_chenran_warmup.fit(model=litgpsr_chenran_warmup, train_dataloaders=train_loader)
print("Chenran loop-based warmup pass completed.\n")

print(f"--- RUN 2: Chenran's Loop-Based Version on {device.upper()} ---")
# Reset seed for exact initialization
torch.manual_seed(42)
np.random.seed(42)
L.seed_everything(42)

# Clear any remaining cache
for elem in gpsr_lattice.segment.elements:
    if hasattr(elem, "_cache"):
        elem._cache.clear()
    if hasattr(elem, "elements"):
        for sub_elem in elem.elements:
            if hasattr(sub_elem, "_cache"):
                sub_elem._cache.clear()

gpsr_model_chenran = GPSR(NNParticleBeamGenerator(10000, p0c), gpsr_lattice)
litgpsr_chenran = LitGPSR(gpsr_model_chenran)

trainer_chenran = L.Trainer(
    max_epochs=max_epochs,
    accelerator=accelerator,
    devices=devices,
    enable_checkpointing=False,
    logger=False
)

t0 = time.perf_counter()
trainer_chenran.fit(model=litgpsr_chenran, train_dataloaders=train_loader)
t_chenran = time.perf_counter() - t0
print(f"\nChenran Loop-based execution time: {t_chenran:.4f} seconds")

In [ ]:
speedup = t_chenran / t_opt
print("\n================ BENCHMARK RESULT ================")
print(f"Chenran's Loop-based version: {t_chenran:.4f} seconds")
print(f"Current Optimized version   : {t_opt:.4f} seconds")
print(f"Speedup                     : {speedup:.4f}x")
print("==================================================")

# Save results
run_data = [{
    "System": system_label,
    "Device": device.upper(),
    "Chenran's Loop-based (s)": t_chenran,
    "Current Optimized (s)": t_opt,
    "Speedup": speedup
}]
df_run = pd.DataFrame(run_data)

system_slug = re.sub(r'[^a-z0-9]+', '_', system_label.lower()).strip('_')
results_dir = os.path.join(repo_root, "dev", "benchmark_results")
os.makedirs(results_dir, exist_ok=True)
csv_filename = os.path.join(results_dir, f"reconstruction_6d_results_{device.lower()}_{system_slug}.csv")
df_run.to_csv(csv_filename, index=False)
print(f"Saved current run results to {csv_filename}")

# Load all reconstruction 6d results files
all_files = glob.glob(os.path.join(results_dir, "reconstruction_6d_results_*.csv"))
print(f"Found {len(all_files)} reconstruction 6D results files:")
all_dfs = []
for file in all_files:
    print(f" - {os.path.basename(file)}")
    all_dfs.append(pd.read_csv(file))
combined_df = pd.concat(all_dfs, ignore_index=True)
print(combined_df)